# 06 - Conclusions

Section **3.6 Conclusions** of the report.

## Brief

Synthesis of results, lessons learned, and possible directions for future work.

- Synthesis of results.
- Lessons learned.
- Future work.

In [1]:
import json

import joblib
import numpy as np
import pandas as pd
import torch
from scipy import sparse

from diplo_mod_1.constants import (
    CONFIGS,
    INTERIM,
    MODELS,
    PRIMARY_CSV,
    PROCESSED,
    RANDOM_STATE,
    RAW,
    REPORTS,
)
from diplo_mod_1.preprocessing.cleaner import DataCleaner
from diplo_mod_1.preprocessing.encoders import TabularEncoder
from diplo_mod_1.preprocessing.feature_engineer import FeatureEngineer
from diplo_mod_1.training.config import TuningHistory
from diplo_mod_1.training.nn_model import WineScoreNet, WineScorePredictorNet

## Step 1 — Synthesis of results

The final numbers already on record across notebooks 03, 04, 04b, and 05, pulled together in one place.

In [2]:
xgb_history = TuningHistory.model_validate_json(
    (REPORTS / "xgboost_metrics.json").read_text(encoding="utf-8")
)
nn_history = TuningHistory.model_validate_json(
    (REPORTS / "nn_metrics.json").read_text(encoding="utf-8")
)
nn_ablation_history = TuningHistory.model_validate_json(
    (REPORTS / "nn_tabular_ablation_metrics.json").read_text(encoding="utf-8")
)


def _test_metrics(run):
    m = next(x for x in run.metrics if x.split == "test")
    return m.rmse, m.r2


rows = []
for label, history in [
    ("XGBoost (44 tabular + 2000 TF-IDF, tuned)", xgb_history),
    ("Neural Net (44 tabular + 2000 TF-IDF, tuned)", nn_history),
    ("Neural Net (44 tabular only, same tuning budget)", nn_ablation_history),
]:
    best = history.best_run(split="test")
    assert best is not None
    rmse, r2 = _test_metrics(best)
    rows.append({"model": label, "test_rmse": rmse, "test_r2": r2})

summary_df = pd.DataFrame(rows)
summary_df

,model,test_rmse,test_r2
0,"XGBoost (44 tabular + 2000 TF-IDF, tuned)",1.445316,0.774511
1,"Neural Net (44 tabular + 2000 TF-IDF, tuned)",1.462726,0.769046
2,"Neural Net (44 tabular only, same tuning budget)",1.869429,0.622760


Three rounds of NN tuning, all on the same full feature set — each gain came from search/regularization changes, not a different model:

| round | config | test RMSE | test R² |
|---|---|---|---|
| Initial search (10 trials) | `nn_training.json` | 1.595 | 0.725 |
| Dense preload + Optuna pruning (20 trials) | `nn_training_wide.json` | 1.479 | 0.764 |
| + activation choice, grad clipping, LR scheduler (30 trials) | `nn_training_wide_v1.json` | 1.463 | 0.769 |

## Step 2 — What the dataset itself limits

Every improvement attempted in this project — wider search, more capable activations, the TF-IDF block, hyperparameter tuning — worked on the same underlying data. Some real limits live in that data itself, not in either model, and no amount of further tuning removes them. Grounded in `notebooks/01-eda.ipynb`'s own findings, with a "what would change if this changed" angle on each:

**Taster bias.** A one-way ANOVA on `points` by `taster_name` gives F=612.05, p≈0, η²=0.099 — reviewer identity alone explains ~10% of the variance in the target. Per-taster means range from 85.86 to 90.56. `taster_avg_points`/`taster_strictness` already correct for a taster's *average* leniency via target encoding, but a given taster's idiosyncrasy on a *specific* review — beyond their own average — isn't modeled and can't be, from a single score per wine. If this dataset had multiple independent tasters scoring the same wines (averaged), that 10% noise floor would shrink directly, and both models would likely close more of the remaining ~0.22-0.23 unexplained-variance gap (1 − R²) than any further architecture or search change could.

**Price.** 6.9% of rows are missing `price` (median-imputed by (country, variety) group, falling back to the global median); the raw distribution has skewness 18.0 and a tail out to $3,300. `log_price` and `price_vs_variety` are consistently among the highest-SHAP-importance features for *both* models (notebook 05 Step 6) — they're doing real work. Cleaner, more complete price data (fewer imputed values, verified retail rather than listed price) would sharpen the single strongest price-derived signal in the whole feature set.

**`region_2`.** 61.1% missing overall, and structurally so — only present for US wines at all, and even there mostly not recoverable from `region_1` (only 368 of 58,213 candidate rows had a clean 1-to-1 mapping). Dropped entirely from the pipeline (`cleaner.py:44`). More consistent sub-region/appellation labeling across *all* countries, not just some US wines, would add a real geographic signal below `region_1`/`province` — both of which already rank highly in SHAP as it stands.

**Vintage survivorship bias.** Pre-1980 vintages score above the dataset average — not because those wines are better, but because only exceptional old bottles are still around (and worth reviewing) decades later. Notebook 01 calls this out explicitly as survivorship bias, not wine chemistry. `wine_age` as currently computed can't distinguish "old and good" from "old because it survived." A dataset with review timing normalized relative to release date (e.g. "reviewed within N years of vintage" for every row, not just recent ones) would remove this confound rather than encode it.

**Long-tail categories.** `winery` alone has 16,757 unique values, most with only a handful of reviews. `winery_freq` and `winery_avg_points` — both used by both models — are exactly the features noisiest for rare categories: a target-encoded average from 2 reviews is a much weaker estimate than one from 200. This also shows up at the country level: Chile (mean 86.49) and Argentina (86.71) score lower than the US/France and have proportionally fewer reviews. More reviews per winery/variety/country — especially for the underrepresented ones — would stabilize the aggregate-encoded features both models lean on most, not just add raw row count.

**Single-source snapshot.** This entire dataset is one publication's (Wine Enthusiast) house style of tasting notes. The curated `TASTING_KEYWORDS` vocabulary and the TF-IDF vectorizer are both fit to that specific writing style ("tannins", "finish", "palate", "nose" — professional tasting-note language, not how a typical consumer would describe a wine). A broader mix of review sources would be a genuine test of whether the text signal generalizes, or is partly an artifact of one publication's vocabulary.

## Step 3 — XGBoost vs. Neural Network: which, and why

**XGBoost is the better model here** — test R² 0.7745 vs. the NN's 0.7690 (test RMSE 1.445 vs. 1.463), and it stayed narrowly ahead through every NN improvement this project tried: dense-data preloading, a wider Optuna search, activation choice, gradient clipping, an LR scheduler. That's the expected outcome for this dataset — ~130k rows of mixed tabular + sparse bag-of-words features is squarely gradient-boosted-tree territory, and GBMs are well known to be hard to beat on structured/tabular data at this scale, especially against a from-scratch MLP with no sequence-aware text modeling.

**But the neural network doesn't perform badly** — 0.769 vs. 0.775 is a ~1.2% relative RMSE gap, not a blowout, and it closed most of that gap over the course of this project without ever changing its feature set or basic architecture family.

**The more consistent finding across this whole project, for *both* models: what actually moved accuracy was the full 2044-column feature set plus hyperparameter tuning — not model architecture choice.** The evidence is already on record, three separate times:

1. **XGBoost's own tabular-only → tabular+TF-IDF jump**: ~10 separate Optuna searches on the 44 tabular columns alone plateaued at test R² ≈ 0.71-0.73 regardless of depth/regularization/learning-rate changes; adding the same 2000-term TF-IDF block the NN uses pushed it to 0.775 in one step.
2. **The NN's tabular-only ablation** (notebook 04b, same tuning budget, same search space, only 44 columns instead of 2044): test R² 0.623 — worse than even the NN's *first, least-tuned* full-feature run (0.725).
3. **The NN's three-round tuning arc** (Step 1's table above): every gain — 0.725 → 0.764 → 0.769 — came from search/regularization/optimizer changes on the exact same feature set and the same basic MLP family. It never needed a structurally different model to close most of its gap with XGBoost; it needed a wider search and a training loop that could actually converge well (dense data loading, pruning, a scheduler).

Put plainly: neither model's default, untuned hyperparameters got anywhere near either model's final tuned numbers. On this dataset, feature richness and tuning budget mattered more than which of these two model families you picked.

## Step 4 — Predict a new wine (synthetic example)

Re-fits `DataCleaner`/`FeatureEngineer` fresh on the full raw dataset and `TabularEncoder` fresh on the train split — all three are deterministic fit/transform classes given the same data and this project's fixed `RANDOM_STATE`, so re-fitting reproduces exactly what produced `data/processed/` the first time; nothing new is invented here, just orchestrating existing, already-tested classes on a brand new row. `scaler.joblib`/`tfidf_vectorizer.joblib` are loaded directly (already-established pattern, no re-fit needed).

In [3]:
cleaner = DataCleaner()
raw_df = DataCleaner.load(RAW / PRIMARY_CSV)
cleaner.fit(raw_df)

engineer = FeatureEngineer()
engineer.fit(raw_df)

split_idx = np.load(PROCESSED / "split_indices.npz")
featured_full = pd.read_parquet(INTERIM / "02_features.parquet")
train_df = featured_full.iloc[split_idx["train"]]
y_train = np.load(PROCESSED / "nn" / "y_train.npy")

tabular_encoder = TabularEncoder()
tabular_encoder.fit(train_df, y_train)

nn_dir = PROCESSED / "nn"
scaler = joblib.load(nn_dir / "scaler.joblib")
tfidf_vectorizer = joblib.load(nn_dir / "tfidf_vectorizer.joblib")

expected_feature_names = json.loads((nn_dir / "feature_names.json").read_text(encoding="utf-8"))[
    "feature_names"
]
continuous_idx = json.loads((nn_dir / "continuous_column_indices.json").read_text(encoding="utf-8"))

print("Re-fitted DataCleaner, FeatureEngineer, and TabularEncoder on-demand.")

Re-fitted DataCleaner, FeatureEngineer, and TabularEncoder on-demand.


Load both best models (same pattern as notebook 05).

In [4]:
xgb_model = joblib.load(MODELS / "xgboost_best.joblib")

checkpoint = torch.load(MODELS / "nn_best.pt", map_location="cpu")
# nn_best.pt predates NNModelRegistry persisting `activation` -- known from
# this session's run to be 'gelu'; new checkpoints self-describe correctly.
checkpoint_activation = checkpoint.get("activation", "gelu")
nn_net = WineScoreNet(
    input_dim=checkpoint["input_dim"],
    hidden_sizes=checkpoint["hidden_sizes"],
    dropout=checkpoint["dropout"],
    activation=checkpoint_activation,
)
nn_net.load_state_dict(checkpoint["state_dict"])
nn_net.eval()

nn_model = WineScorePredictorNet(
    input_dim=checkpoint["input_dim"],
    hidden_sizes=checkpoint["hidden_sizes"],
    dropout=checkpoint["dropout"],
    activation=checkpoint_activation,
    device="cpu",
)
nn_model.model_ = nn_net

print(f"XGBoost n_features_in_={xgb_model.n_features_in_}, NN activation={checkpoint_activation}")

XGBoost n_features_in_=2044, NN activation=gelu


The transform: `clean` → `engineer` → `tabular_encoder` (same three-call pipeline notebook 02 uses at training time), then vectorize the description and assemble each model's expected input shape (XGBoost: unscaled tabular + TF-IDF; NN: scaled tabular + TF-IDF, densified).

In [5]:
def raw_wine_to_model_inputs(wine: dict):
    """Turn one raw wine record into (xgb_input, nn_input), matching each
    model's expected 2044-column representation exactly.

    `wine` needs the raw CSV schema: country, description, designation,
    price, province, region_1, taster_name, title (must contain a 4-digit
    year, e.g. "Some Wine 2021" -- used to derive wine_age), variety, winery.
    `price`/`designation`/`region_1`/`taster_name` may be None/omitted; they
    fall back the same way training data with missing values did.
    """
    row = pd.DataFrame([wine])
    cleaned = cleaner.clean(row)
    featured = engineer.transform(cleaned)
    tab = tabular_encoder.transform(featured)
    assert tab.feature_names == expected_feature_names, (
        "re-fitted TabularEncoder's column order doesn't match feature_names.json "
        "-- something upstream has drifted from notebook 02's pipeline"
    )

    txt_row = tfidf_vectorizer.transform(featured["description"].astype(str))

    xgb_input = sparse.hstack([sparse.csr_matrix(tab.X), txt_row], format="csr")

    tab_scaled = tab.X.copy()
    tab_scaled[:, continuous_idx] = scaler.transform(tab.X[:, continuous_idx])
    nn_input = (
        sparse.hstack([sparse.csr_matrix(tab_scaled), txt_row], format="csr")
        .toarray()
        .astype(np.float32)
    )

    return xgb_input, nn_input


def predict_wine(wine: dict) -> None:
    xgb_input, nn_input = raw_wine_to_model_inputs(wine)
    xgb_pred = xgb_model.predict(xgb_input)[0]
    nn_pred = nn_model.predict(nn_input)[0]
    print(f"{wine['winery']} {wine['variety']} ({wine['country']}), ${wine['price']}")
    print(f"  XGBoost predicted points: {xgb_pred:.1f}")
    print(f"  Neural Net predicted points: {nn_pred:.1f}")
    print()

Three synthetic examples spanning the price/quality range.

In [6]:
everyday_wine = {
    "country": "US",
    "description": (
        "A simple, easy-drinking red with light cherry and berry notes and a "
        "soft, short finish. Nothing fancy, decent for the price."
    ),
    "designation": None,
    "price": 12.0,
    "province": "California",
    "region_1": "Central Valley",
    "taster_name": "Unknown",
    "title": "Example Everyday Red 2021",
    "variety": "Merlot",
    "winery": "Sunny Valley Cellars",
}

mid_range_wine = {
    "country": "Argentina",
    "description": (
        "A rich, ripe Malbec with dark berry and cherry notes, hints of black "
        "pepper and vanilla oak, medium tannins and a smooth, lasting finish."
    ),
    "designation": "Reserva",
    "price": 28.0,
    "province": "Mendoza Province",
    "region_1": "Lujan de Cuyo",
    "taster_name": "Michael Schachner",
    "title": "Example Winery Malbec Reserva 2019",
    "variety": "Malbec",
    "winery": "Example Winery",
}

luxury_wine = {
    "country": "France",
    "description": (
        "An exceptional, complex wine with layers of dark fruit, spice, "
        "graphite and violet, remarkably concentrated yet elegant, with a "
        "long, polished, structured finish. Aromas of cassis and cedar "
        "linger on the nose. A profound, age-worthy bottling."
    ),
    "designation": "Grand Cru",
    "price": 250.0,
    "province": "Bordeaux",
    "region_1": "Pauillac",
    "taster_name": "Roger Voss",
    "title": "Example Chateau Grand Cru 2016",
    "variety": "Bordeaux-style Red Blend",
    "winery": "Example Chateau",
}

for wine in (everyday_wine, mid_range_wine, luxury_wine):
    predict_wine(wine)

c:\Users\leona\source\repos\diplo-mod-1\.venv\lib\site-packages\xgboost\core.py:751: UserWarning: [22:58:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Sunny Valley Cellars Merlot (US), $12.0
  XGBoost predicted points: 83.6
  Neural Net predicted points: 84.9

Example Winery Malbec (Argentina), $28.0
  XGBoost predicted points: 86.8
  Neural Net predicted points: 87.7

Example Chateau Bordeaux-style Red Blend (France), $250.0
  XGBoost predicted points: 94.6
  Neural Net predicted points: 91.8



Edit freely and re-run — any country/variety/winery/taster not seen during training falls back gracefully (target encoders return their fitted global mean for unseen categories; frequency maps return 0).

In [ ]:
your_wine = {
    "country": "Argentina",
    "description": "Describe the wine here -- try different tasting notes and see how the predictions move.",
    "designation": None,
    "price": 18.0,
    "province": "Central Valley",
    "region_1": None,
    "taster_name": "Unknown",
    "title": "Your Wine 2022",
    "variety": "Cabernet Sauvignon",
    "winery": "Your Winery",
}

predict_wine(your_wine)

## Step 5 — Lessons learned & future work

**Modeling:**
- Replace the TF-IDF block with a frozen pretrained sentence-embedding (e.g. `sentence-transformers`) — same "concatenate as another dense block" architecture both models already use, no RNN/Transformer training required, likely a stronger and more efficient text representation than TF-IDF.
- Keep widening the NN's hyperparameter search now that it's cheap (dense preloading + Optuna pruning cut wall-clock substantially) — more trials, per-layer dropout, independently-searched depth/width instead of fixed architecture strings.
- Exclude BatchNorm/bias parameters from weight decay — a well-known best practice not yet applied; AdamW currently regularizes every NN parameter uniformly.
- A cheap ensemble (blend/average XGBoost + NN predictions) — the two models reach similar accuracy through different mechanisms (individual TF-IDF words vs. tabular aggregates, per notebook 05's SHAP comparison), so their errors may be meaningfully decorrelated.

**Data** (from Step 2 above, reframed as concrete next steps):
- More reviewers per wine, averaged, to shrink the ~10%-of-variance taster-bias noise floor directly.
- Consistent sub-region/appellation labels across all countries, not just some US wines, to extend the `region_1`/`province` signal both models already rely on.
- Review timing normalized relative to vintage/release date, to remove the vintage survivorship-bias confound rather than encode it.
- More reviews for underrepresented countries/varieties/wineries specifically — not just more rows overall — to stabilize the target- and frequency-encoded features that are noisiest exactly where data is thinnest.
- A second review-text source, to test whether the text signal generalizes beyond one publication's tasting-note vocabulary.